In [3]:
!mkdir -p /kaggle/working/tokenizer_data

!kaggle datasets download \
    -d mathimohankowshik/tokeenizer \
    -p /kaggle/working/tokenizer_data \
    --unzip

print("\nFiles downloaded:")
!find /kaggle/working/tokenizer_data -maxdepth 2 -type f -printf "%p\n"

Dataset URL: https://www.kaggle.com/datasets/mathimohankowshik/tokeenizer
License(s): unknown
100%|█████████████████████████████████████████| 657k/657k [00:00<00:00, 925kB/s]


Files downloaded:
/kaggle/working/tokenizer_data/tokenizer.json


In [2]:
!pip -q install -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 1.1 MB/s eta 0:00:00a 0:00:01


In [4]:
from pathlib import Path
from tokenizers import Tokenizer

TOKENIZER_PATH = Path("/kaggle/working/tokenizer_data/tokenizer.json")

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Tokenizer loaded:", TOKENIZER_PATH.exists())
print("File size:", f"{TOKENIZER_PATH.stat().st_size / 1024:.2f} KB")
print("Vocab size:", tokenizer.get_vocab_size())

print("\nSpecial tokens:")
for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
    print(f"{token}: {tokenizer.token_to_id(token)}")

text = "The petitioner filed an appeal before the Supreme Court."
encoded = tokenizer.encode(text)

print("\nTest text:", text)
print("Token count:", len(encoded.ids))
print("Token IDs:", encoded.ids)

Tokenizer loaded: True
File size: 3447.88 KB
Vocab size: 50000

Special tokens:
[PAD]: 0
[UNK]: 1
[BOS]: 2
[EOS]: 3
[MASK]: 4

Test text: The petitioner filed an appeal before the Supreme Court.
Token count: 10
Token IDs: [550, 436, 593, 349, 668, 647, 238, 1487, 430, 18]


In [5]:
# A2 configuration

VOCAB_SIZE = tokenizer.get_vocab_size()

D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 8
FFN_DIM = 2048
CONTEXT_LENGTH = 2048

DROPOUT = 0.1

print("A2 configuration")
print("-----------------")
print("Vocabulary:", VOCAB_SIZE)
print("Hidden size:", D_MODEL)
print("Attention heads:", NUM_HEADS)
print("Layers:", NUM_LAYERS)
print("FFN dimension:", FFN_DIM)
print("Context length:", CONTEXT_LENGTH)
print("Dropout:", DROPOUT)

A2 configuration
-----------------
Vocabulary: 50000
Hidden size: 512
Attention heads: 8
Layers: 8
FFN dimension: 2048
Context length: 2048
Dropout: 0.1


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU count:", torch.cuda.device_count())

class A2GQATransformer(nn.Module):
    """
    A2: Decoder-only Transformer with Grouped-Query Attention.

    Fill in the GQA attention implementation and block logic
    according to your own A2 design.
    """

    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        ffn_dim=FFN_DIM,
        context_length=CONTEXT_LENGTH,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.ffn_dim = ffn_dim
        self.context_length = context_length
        self.dropout = dropout

        # TODO:
        # 1. Token embeddings
        # 2. Positional representation
        # 3. Your GQA Transformer blocks
        # 4. Final normalization
        # 5. Output projection

        raise NotImplementedError(
            "Implement your A2 GQA architecture here."
        )

Device: cuda
GPU count: 2


In [7]:
import os
import torch

# ==============================
# A2 — EFFICIENT GQA
# ==============================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VOCAB_SIZE = 50_000
CONTEXT_LEN = 2048

HIDDEN_SIZE = 528
NUM_LAYERS = 8
NUM_Q_HEADS = 8
NUM_KV_HEADS = 2
FFN_SIZE = 2048

TRAIN_SEQ_LEN = 512
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.1
TOKEN_BUDGET = 200_000_000

TOKENIZER_PATH = "/kaggle/input/datasets/mohan567/tokeni/tokenizer.json"

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

CHECKPOINT_DIR = "/kaggle/working/a2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("A2 — Efficient GQA")
print("Device :", DEVICE)

if DEVICE == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("Parameters target : ~50.41M")
print("Token budget      :", TOKEN_BUDGET)
print("Train sequence    :", TRAIN_SEQ_LEN)

A2 — Efficient GQA
Device : cuda
GPU    : Tesla T4
VRAM   : 14.56 GB
Parameters target : ~50.41M
Token budget      : 200000000
Train sequence    : 512


In [15]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded")
print("Vocabulary size:", tokenizer.get_vocab_size())
print("PAD:", tokenizer.token_to_id("[PAD]"))
print("UNK:", tokenizer.token_to_id("[UNK]"))
print("BOS:", tokenizer.token_to_id("[BOS]"))
print("EOS:", tokenizer.token_to_id("[EOS]"))

Exception: No such file or directory (os error 2)

In [16]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "tokenizer.json":
            print(os.path.join(root, f))

/kaggle/input/datasets/mathimohankowshik/tokenizer/tokenizer.json


In [17]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/mathimohankowshik/tokenizer/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded")
print("Vocabulary size:", tokenizer.get_vocab_size())
print("PAD:", tokenizer.token_to_id("[PAD]"))
print("UNK:", tokenizer.token_to_id("[UNK]"))
print("BOS:", tokenizer.token_to_id("[BOS]"))
print("EOS:", tokenizer.token_to_id("[EOS]"))

Tokenizer loaded
Vocabulary size: 50000
PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [18]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(
            x.pow(2).mean(-1, keepdim=True) + self.eps
        ) * self.weight


class GQAAttention(nn.Module):
    def __init__(self, hidden_size, num_q_heads, num_kv_heads):
        super().__init__()

        assert hidden_size % num_q_heads == 0

        self.hidden_size = hidden_size
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_q_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(
            hidden_size,
            num_kv_heads * self.head_dim
        )
        self.v_proj = nn.Linear(
            hidden_size,
            num_kv_heads * self.head_dim
        )
        self.o_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_q_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # Repeat KV heads for GQA
        repeat_factor = self.num_q_heads // self.num_kv_heads
        k = k.repeat_interleave(repeat_factor, dim=1)
        v = v.repeat_interleave(repeat_factor, dim=1)

        y = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.o_proj(y)


class GQABlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.norm1 = RMSNorm(HIDDEN_SIZE)
        self.attn = GQAAttention(
            HIDDEN_SIZE,
            NUM_Q_HEADS,
            NUM_KV_HEADS
        )

        self.norm2 = RMSNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.SiLU(),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class A2GQAModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LEN,
            HIDDEN_SIZE
        )

        self.layers = nn.ModuleList(
            [GQABlock() for _ in range(NUM_LAYERS)]
        )

        self.norm = RMSNorm(HIDDEN_SIZE)

        # Tied input/output embeddings
        self.lm_head = nn.Linear(
            HIDDEN_SIZE,
            VOCAB_SIZE,
            bias=False
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1)
            )

        return logits, loss


model = A2GQAModel().to(DEVICE)

parameter_count = sum(
    p.numel() for p in model.parameters()
)

print("A2 parameters:", parameter_count)
print("A2 parameters (M):", parameter_count / 1e6)

A2 parameters: 50398672
A2 parameters (M): 50.398672


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(
            x.pow(2).mean(dim=-1, keepdim=True) + self.eps
        ) * self.weight


class GQAAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.head_dim = HIDDEN_SIZE // NUM_Q_HEADS

        self.q_proj = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE,
            bias=True
        )

        self.k_proj = nn.Linear(
            HIDDEN_SIZE,
            NUM_KV_HEADS * self.head_dim,
            bias=True
        )

        self.v_proj = nn.Linear(
            HIDDEN_SIZE,
            NUM_KV_HEADS * self.head_dim,
            bias=True
        )

        self.o_proj = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE,
            bias=True
        )

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T, NUM_Q_HEADS, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T, NUM_KV_HEADS, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T, NUM_KV_HEADS, self.head_dim
        ).transpose(1, 2)

        # GQA: expand KV heads to match Q heads
        repeat_factor = NUM_Q_HEADS // NUM_KV_HEADS

        k = k.repeat_interleave(
            repeat_factor,
            dim=1
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1
        )

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True
        )

        y = y.transpose(1, 2).contiguous()

        y = y.view(B, T, C)

        return self.o_proj(y)


class GQABlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = RMSNorm(HIDDEN_SIZE)
        self.attn = GQAAttention()

        self.ln2 = RMSNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.SiLU(),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class A2GQAModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LEN,
            HIDDEN_SIZE
        )

        self.layers = nn.ModuleList(
            [
                GQABlock()
                for _ in range(NUM_LAYERS)
            ]
        )

        self.ln_f = RMSNorm(HIDDEN_SIZE)

        self.lm_head = nn.Linear(
            HIDDEN_SIZE,
            VOCAB_SIZE,
            bias=False
        )

        # Weight tying
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self, input_ids, targets=None):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        for layer in self.layers:
            x = layer(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1)
            )

        return logits, loss


model = A2GQAModel().to(DEVICE)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

print("A2 parameters:", parameter_count)
print("A2 parameters (M):", parameter_count / 1e6)

A2 parameters: 50398672
A2 parameters (M): 50.398672


In [26]:
# ============================================================
# A2 — EFFICIENT GQA : PRODUCTION TRAINING
# ============================================================

import os
import json
import time
import requests
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_url, get_token

# -----------------------------
# Configuration
# -----------------------------

HF_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

TOKEN_BUDGET = 200_000_000
SEQ_LEN = 512

BATCH_SIZE = 2
GRAD_ACCUM = 4

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

CHECKPOINT_EVERY = 25_000

CHECKPOINT_DIR = "/kaggle/working/a2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# -----------------------------
# Speed settings
# -----------------------------

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda")

# -----------------------------
# HF authentication
# -----------------------------

HF_TOKEN = get_token()

HEADERS = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

# ============================================================
# JSON DATA STREAM
# ============================================================

def open_json_source(filename):

    url = hf_hub_url(
        repo_id=HF_REPO,
        filename=filename,
        repo_type="dataset"
    )

    response = requests.get(
        url,
        headers=HEADERS,
        stream=True,
        timeout=120
    )

    response.raise_for_status()

    return response


def extract_texts(obj):

    if isinstance(obj, str):
        return [obj]

    if isinstance(obj, list):
        result = []

        for item in obj:
            result.extend(extract_texts(item))

        return result

    if isinstance(obj, dict):

        result = []

        for key in ["text", "content", "body"]:
            if key in obj and isinstance(obj[key], str):
                result.append(obj[key])

        if result:
            return result

        for value in obj.values():
            result.extend(extract_texts(value))

        return result

    return []


def load_source(filename):

    response = open_json_source(filename)

    data = response.json()

    texts = extract_texts(data)

    response.close()

    return texts


# ============================================================
# TOKEN BLOCK GENERATOR
# ============================================================

def training_blocks():

    source_index = 0

    token_buffer = []

    while True:

        filename = TRAIN_FILES[source_index]

        texts = load_source(filename)

        source_index = (
            source_index + 1
        ) % len(TRAIN_FILES)

        for text in texts:

            encoded = tokenizer.encode(text).ids

            if len(encoded) < 2:
                continue

            token_buffer.extend(encoded)

            while len(token_buffer) >= SEQ_LEN + 1:

                block = token_buffer[:SEQ_LEN + 1]

                token_buffer = token_buffer[SEQ_LEN:]

                yield (
                    torch.tensor(
                        block[:-1],
                        dtype=torch.long
                    ),
                    torch.tensor(
                        block[1:],
                        dtype=torch.long
                    )
                )


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY,
    fused=True
)

# ============================================================
# AMP
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

# ============================================================
# TRAINING
# ============================================================

model.train()

blocks_per_step = BATCH_SIZE

tokens_per_step = (
    BATCH_SIZE
    * GRAD_ACCUM
    * SEQ_LEN
)

total_steps = (
    TOKEN_BUDGET // tokens_per_step
)

data = training_blocks()

optimizer.zero_grad(set_to_none=True)

start_time = time.time()

tokens_seen = 0

for step in range(1, total_steps + 1):

    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM):

        inputs = []
        targets = []

        for _ in range(BATCH_SIZE):

            x, y = next(data)

            inputs.append(x)
            targets.append(y)

        x = torch.stack(inputs).to(
            DEVICE,
            non_blocking=True
        )

        y = torch.stack(targets).to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            _, loss = model(
                x,
                y
            )

            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        accumulated_loss += loss.item()

        tokens_seen += (
            BATCH_SIZE * SEQ_LEN
        )

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    scaler.step(optimizer)

    scaler.update()

    optimizer.zero_grad(
        set_to_none=True
    )

    # -------------------------
    # Checkpoint
    # -------------------------

    if step % CHECKPOINT_EVERY == 0:

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"step_{step:06d}.pt"
        )

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "step": step,
                "tokens_seen": tokens_seen,
                "config": {
                    "vocab_size": VOCAB_SIZE,
                    "context_len": CONTEXT_LEN,
                    "hidden_size": HIDDEN_SIZE,
                    "num_layers": NUM_LAYERS,
                    "num_q_heads": NUM_Q_HEADS,
                    "num_kv_heads": NUM_KV_HEADS,
                    "ffn_size": FFN_SIZE,
                    "train_seq_len": SEQ_LEN
                }
            },
            checkpoint_path
        )


# ============================================================
# FINAL CHECKPOINT
# ============================================================

final_path = os.path.join(
    CHECKPOINT_DIR,
    "a2_final.pt"
)

torch.save(
    {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "step": total_steps,
        "tokens_seen": tokens_seen,
        "config": {
            "vocab_size": VOCAB_SIZE,
            "context_len": CONTEXT_LEN,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "num_q_heads": NUM_Q_HEADS,
            "num_kv_heads": NUM_KV_HEADS,
            "ffn_size": FFN_SIZE,
            "train_seq_len": SEQ_LEN
        }
    },
    final_path
)

elapsed = time.time() - start_time

print("A2 TRAINING COMPLETE")
print("Steps :", total_steps)
print("Tokens:", tokens_seen)
print("Time  :", round(elapsed / 3600, 2), "hours")
print("Model :", final_path)

HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_acts.json

In [22]:
from huggingface_hub import HfApi, get_token

HF_TOKEN = get_token()

print("HF token available:", HF_TOKEN is not None)

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="Exploration-Lab/CS-781-Capstone",
    repo_type="dataset"
)

TRAIN_FILES = [
    f for f in files
    if f.startswith("Phase_1_Pretraining/Train/")
]

print("Training files found:")
for f in TRAIN_FILES:
    print(f)

HF token available: False
Training files found:
Phase_1_Pretraining/Train/train_acts.json
Phase_1_Pretraining/Train/train_high_courts.json
Phase_1_Pretraining/Train/train_supreme_court.json
Phase_1_Pretraining/Train/train_tribunals.json


In [25]:
from huggingface_hub import hf_hub_download, get_token

HF_TOKEN = get_token()

print("HF token available:", HF_TOKEN is not None)

path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Train/train_acts.json",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded:", path)

HF token available: False


GatedRepoError: 401 Client Error. (Request ID: Root=1-6ac30e9f-4edac2a30045d4f575763d35;f1be40ba-e5b8-4823-bec2-2bca54d3be40)

Cannot access gated repo for url https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_acts.json.
Access to dataset Exploration-Lab/CS-781-Capstone is restricted. You must have access to it and be authenticated to access it. Please log in.

In [27]:
from huggingface_hub import get_token, HfApi

HF_TOKEN = get_token()

print("Token available:", HF_TOKEN is not None)

api = HfApi(token=HF_TOKEN)

info = api.dataset_info(
    "Exploration-Lab/CS-781-Capstone"
)

print("Dataset access: OK")
print("Dataset:", info.id)

Token available: False
Dataset access: OK
Dataset: Exploration-Lab/CS-781-Capstone


In [31]:
from huggingface_hub import hf_hub_download

def load_source(filename):
    path = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename=filename,
        repo_type="dataset"
    )

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [32]:
test_file = "Phase_1_Pretraining/Train/train_acts.json"

texts = load_source(test_file)

print("Acts loaded successfully")
print("Type:", type(texts))

GatedRepoError: 401 Client Error. (Request ID: Root=1-6ac30f79-3ac55fe35019ab7a619c783e;493ecaa2-642e-47d2-b2ac-47e9c2714d4a)

Cannot access gated repo for url https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_acts.json.
Access to dataset Exploration-Lab/CS-781-Capstone is restricted. You must have access to it and be authenticated to access it. Please log in.

In [1]:
from pathlib import Path

for name in [
    "prediction.json",
    "submission.zip",
    "phase1_a2.zip",
]:
    matches = list(Path("/kaggle/working").rglob(name))
    print(f"\n{name}:")
    if matches:
        for path in matches[:10]:
            print(" ", path)
    else:
        print("  Not found")


prediction.json:
  Not found

submission.zip:
  Not found

phase1_a2.zip:
  Not found
